# Module 3 Practical — The API Engine Room 🔌

**PMS RoBoTics Research Center · 4-Week Internship & Training Program**
**Week 1 · Module 3: Working with LLM APIs — OpenAI, Anthropic, and open-source models**

---

### What you'll do in the next 30 minutes

| # | Experiment | Skill |
|---|-----------|-------|
| 1 | First OpenAI call | The client → model + messages → text pattern |
| 2 | Build memory by hand | The stateless API & the messages list |
| 3 | Port SmartBot v2 | Prompts are portable across providers |
| 4 | Read the bill | `response.usage` and cost per call |
| 5 | Survive failures | try/except, retries, timeouts |
| 🏁 | **SmartBot v3** | A production-shaped chatbot class |

> ⚙️ **Today's engine: OpenAI.** Days 1–2 used Google Gemini's free tier. Today we use the paid-API workflow professionals use daily — same pattern, different SDK. Your prompts from yesterday work unchanged; that's the whole point of this module.

## Step 0 — Setup 🔑

Your instructor will tell you which key to use (a class key with a spend limit, or your own from [platform.openai.com](https://platform.openai.com/api-keys)).

**Key hygiene (from today's slides):** we enter it with `getpass` — never paste a key into a code cell, never commit one to GitHub.

In [ ]:
%pip install -q -U openai

In [ ]:
from getpass import getpass
from openai import OpenAI

API_KEY = getpass("Paste the OpenAI API key and press Enter: ")
client = OpenAI(api_key=API_KEY)

# Budget-tier default. Model names change often — see the next cell
# to list what YOUR key can access, and swap freely: the code won't change.
MODEL = "gpt-4o-mini"

print("✅ Client ready! Using model:", MODEL)

In [ ]:
# Which models can this key use? (handy whenever names change)
models = client.models.list()
print("A few available models:")
for m in sorted(m.id for m in models.data)[:20]:
    print(" -", m)

---
## Experiment 1 — Your first OpenAI call 🚀

Compare with Day 1's Gemini call: different names, same three moves —
**make a client → send model + messages → read the text.**

New here: the **messages list** with **roles**. Even a single question is a list.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant for engineering students."},
        {"role": "user", "content": "Explain in 2 sentences what an API is, using a restaurant analogy."},
    ],
)

print(response.choices[0].message.content)

**Map it to Gemini in your head:**

| Concept | Gemini (Days 1–2) | OpenAI (today) |
|---|---|---|
| Client | `genai.Client(api_key=...)` | `OpenAI(api_key=...)` |
| Call | `client.models.generate_content(...)` | `client.chat.completions.create(...)` |
| System prompt | `config=...system_instruction` | first message with `"role": "system"` |
| Answer text | `response.text` | `response.choices[0].message.content` |

Five lines of difference. Everything you know still applies.

**✏️ Your turn:** change the system message to one of yesterday's personas (the poet? the impatient store manager?) and re-run.

---
## Experiment 2 — The API has no memory. Build it. 🧠

On Day 1, `client.chats.create(...)` gave us memory for free. OpenAI's chat API doesn't pretend: **every call is stateless**. Watch it forget:

In [ ]:
# Call 1
r1 = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "My name is Soham and I love robotics."}],
)
print("Reply 1:", r1.choices[0].message.content[:120], "...")

# Call 2 — a brand new request. Does it remember?
r2 = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What is my name?"}],
)
print("\nReply 2:", r2.choices[0].message.content)

It has no idea. **"Memory" in every chatbot you've ever used is just this:** the app resends the whole conversation with every call. Let's do exactly that:

In [ ]:
history = [
    {"role": "system", "content": "You are a friendly assistant. Be brief."},
]

def chat(user_text):
    history.append({"role": "user", "content": user_text})
    response = client.chat.completions.create(model=MODEL, messages=history)
    answer = response.choices[0].message.content
    # store the model's reply too — that's what makes turn 3 understand turn 1
    history.append({"role": "assistant", "content": answer})
    return answer

print(chat("My name is Soham and I love robotics."))
print(chat("What is my name, and what do I love?"))
print(f"\n(history now holds {len(history)} messages — it grows every turn!)")

🎉 **You just implemented chat memory from scratch** — the thing Gemini's `chats` object was doing for you secretly.

**Think about it:** the history grows every turn, and you pay for every token of it on every call. What happens after 200 turns? (Hold that thought — context windows are tomorrow's topic, and trimming history is a classic technique.)

---
## Experiment 3 — Port SmartBot v2 to OpenAI 🚚

The portability test. Below is **yesterday's SmartBot v2 system prompt, completely unchanged** — we only swap the engine underneath it.

In [ ]:
SMARTBOT_V2 = """# ROLE
You are SmartBot, the customer assistant of SmartMart retail store, Pimple Saudagar, Pune.

# CONTEXT (the only facts you know — never invent others)
- Store hours: 9 AM - 9 PM, all 7 days
- Returns: within 7 days WITH receipt; without receipt -> store credit only
- Delivery: free above Rs. 999, else Rs. 49; standard 2-4 days in Pune
- Current offer: 10% off kitchen appliances till Sunday
- Contact for escalation: pmssupport@smartmart.example / (+91) 9960664674

# TASK
Answer customer questions about the store, orders, deliveries and policies.

# FORMAT
- Maximum 3 sentences per reply
- If the customer sounds angry, first sentence must acknowledge their frustration

# CONSTRAINTS
- If you don't know something (e.g. live stock, order tracking), say exactly:
  "I'll need to check our system for that — may I have your order number?"
- Politely refuse anything unrelated to SmartMart.
- Never invent prices, stock levels or delivery dates.
"""

smartbot_history = [{"role": "system", "content": SMARTBOT_V2}]

def smartbot(user_text):
    smartbot_history.append({"role": "user", "content": user_text})
    r = client.chat.completions.create(model=MODEL, messages=smartbot_history)
    answer = r.choices[0].message.content
    smartbot_history.append({"role": "assistant", "content": answer})
    return answer

# Yesterday's acceptance tests, now on a different engine:
for t in ["when do you close on sunday?",
          "my kettle broke after 2 days, i'm furious!",
          "do you have iPhone 15 in stock right now?",
          "can you write my physics homework?"]:
    print(f"🧑 {t}")
    print(f"🛒 {smartbot(t)}")
    print("-" * 70)

**Same personality, same rules, different engine.** This is why we spent yesterday on prompts: they are portable assets. Companies switch providers over a weekend when pricing changes — their prompts move with them.

**✏️ Your turn:** does the OpenAI version behave differently from the Gemini version anywhere? Try your best Day-2 jailbreak on it and compare notes.

---
## Experiment 4 — Read the bill 💰

Every response carries its own receipt: `response.usage`. Professionals read it on **every** call.

In [ ]:
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "system", "content": SMARTBOT_V2},
              {"role": "user", "content": "Is delivery free for a Rs. 1,500 mixer?"}],
)

u = r.usage
print("Input (prompt) tokens:    ", u.prompt_tokens)
print("Output (completion) tokens:", u.completion_tokens)
print("Total tokens:              ", u.total_tokens)

In [ ]:
# Turn tokens into rupees.
# ⚠️ RATES CHANGE — check https://platform.openai.com/pricing and update these:
INPUT_RATE_PER_1M  = 0.15   # USD per 1M input tokens  (typical budget-tier rate)
OUTPUT_RATE_PER_1M = 0.60   # USD per 1M output tokens
USD_TO_INR = 90.0           # update to today's rate

cost_usd = (u.prompt_tokens * INPUT_RATE_PER_1M + u.completion_tokens * OUTPUT_RATE_PER_1M) / 1_000_000
print(f"This call cost ≈ ${cost_usd:.6f} = ₹{cost_usd * USD_TO_INR:.4f}")

# scale it up like a real product:
daily = cost_usd * USD_TO_INR * 1000 * 5     # 1,000 customers x ~5 exchanges
print(f"1,000 customers x 5 messages/day ≈ ₹{daily:,.0f}/day")

**Notice where the tokens went:** the system prompt is re-sent with *every* message — that's the price of statelessness. A 400-token system prompt at 5,000 calls/day is 2M tokens/day *before anyone says hello*. This is why yesterday's homework asked you to shrink the prompt without losing rules.

**✏️ Your turn:** re-run the two cells with a one-word question vs a long paragraph question. Watch the input tokens move.

---
## Experiment 5 — Survive failures 🛡️

Amateur code assumes the API always answers. Let's write code that assumes it sometimes won't — and prove it works by triggering a real error:

In [ ]:
# First, trigger a REAL error on purpose: a model that doesn't exist.
try:
    client.chat.completions.create(
        model="gpt-99-ultra",   # nope
        messages=[{"role": "user", "content": "hello"}],
    )
except Exception as e:
    print(type(e).__name__, "->", str(e)[:140])

In [ ]:
import time
import random
from openai import RateLimitError, APITimeoutError, APIError, AuthenticationError

def robust_chat(messages, model=MODEL, max_retries=3, timeout=30):
    """Call the API with retries for transient errors, fail fast for fatal ones."""
    for attempt in range(1, max_retries + 1):
        try:
            r = client.chat.completions.create(
                model=model, messages=messages, timeout=timeout,
            )
            return r.choices[0].message.content, r.usage
        except AuthenticationError:
            raise RuntimeError("Bad API key — fix the key, retrying won't help.")
        except (RateLimitError, APITimeoutError, APIError) as e:
            if attempt == max_retries:
                return ("Sorry, our assistant is busy right now — please try again "
                        "in a minute."), None            # graceful degradation
            wait = (2 ** attempt) + random.random()      # exponential backoff + jitter
            print(f"  [attempt {attempt} failed: {type(e).__name__} — retrying in {wait:.1f}s]")
            time.sleep(wait)

answer, usage = robust_chat([{"role": "user", "content": "Say 'resilient!' and nothing else."}])
print(answer)

**Read the pattern, it's a classic:**
1. **Fatal errors** (bad key) → fail fast with a clear message. Retrying can't fix it.
2. **Transient errors** (rate limit, timeout, server hiccup) → retry with **exponential backoff** (2s, 4s, 8s…) plus **jitter** (randomness so 1,000 clients don't all retry at the same instant).
3. **Out of retries** → degrade gracefully: the customer gets a polite message, never a stack trace.

**✏️ Your turn:** set `max_retries=1` and `timeout=0.001` and watch the graceful degradation path fire.

---
## 🏁 Finale — SmartBot v3

Everything from today in one class: engineered prompt (Day 2), hand-built memory (Exp 2), robust calls (Exp 5), and a live cost meter (Exp 4).

In [ ]:
class SmartBot:
    """SmartMart's assistant, v3 — production-shaped."""

    def __init__(self, system_prompt=SMARTBOT_V2, model=MODEL):
        self.model = model                      # swap engines with one argument
        self.history = [{"role": "system", "content": system_prompt}]
        self.total_prompt_tokens = 0
        self.total_completion_tokens = 0

    def say(self, user_text):
        self.history.append({"role": "user", "content": user_text})
        answer, usage = robust_chat(self.history, model=self.model)
        self.history.append({"role": "assistant", "content": answer})
        if usage:
            self.total_prompt_tokens += usage.prompt_tokens
            self.total_completion_tokens += usage.completion_tokens
        return answer

    def bill(self, usd_to_inr=USD_TO_INR):
        cost = (self.total_prompt_tokens * INPUT_RATE_PER_1M +
                self.total_completion_tokens * OUTPUT_RATE_PER_1M) / 1_000_000
        return (f"{len(self.history)//2} exchanges | "
                f"{self.total_prompt_tokens} in / {self.total_completion_tokens} out tokens | "
                f"≈ ₹{cost * usd_to_inr:.3f}")

bot = SmartBot()
print("🛒", bot.say("hi! till when are you open today?"))
print("🛒", bot.say("great — and can I return something I bought 5 days ago?"))
print("🛒", bot.say("what if I lost the receipt?"))
print()
print("🧾 Conversation bill:", bot.bill())

In [ ]:
# 💬 Chat with v3 yourself — type 'bill' for the cost meter, 'quit' to stop.
while True:
    user_msg = input("You: ")
    cmd = user_msg.strip().lower()
    if cmd in ("quit", "exit", "q"):
        print("👋 Final bill:", bot.bill())
        break
    if cmd == "bill":
        print("🧾", bot.bill())
        continue
    print("🛒 SmartBot:", bot.say(user_msg))

---
## 🏆 Homework (15 min, counts toward your weekly project)

1. **Cost report** — run a 10-message conversation with SmartBot v3, then write down: total tokens, cost in ₹, and what % of input tokens the system prompt is responsible for (hint: system prompt tokens × number of calls).
2. **Engine swap on paper** — using the Experiment 1 mapping table, rewrite `robust_chat` for Gemini (don't run it if you're out of free quota — just write it). What changes? What doesn't?
3. **Memory limit** — add a `MAX_TURNS` cap to the `SmartBot` class: when history exceeds 20 messages, drop the oldest user/assistant pair (never the system message!). This becomes essential tomorrow.

### What's next
**Module 4 — Tokens, temperature, context windows & parameters:** the `temperature`, `max_tokens` and other knobs we deliberately didn't touch today. You'll make the same prompt boring, creative, and unhinged — on purpose.

---
*PMS RoBoTics Research Center · pmsroboticsrc@gmail.com · (+91) 9960664674*